# **4. Базовые модели и метрики качества (Baseline Models and Evaluation Metrics): формирование исходного уровня прогнозирования**

* __Цель:__ обучить набор базовых регрессионных моделей и сравнить их качество на хронологической validation-выборке.
* __Задачи:__
  - воспроизвести загрузку, feature engineering и хронологическое разбиение;
  - обучить preprocessing только на train-выборке;
  - обучить naive и baseline-регрессоры;
  - рассчитать `MAE`, `RMSE`, `MAPE` и `R²` на validation-выборке;
  - сохранить локальную таблицу baseline-метрик.
* __Алгоритм выполнения:__
  1. Загрузить подготовленные данные и сформировать признаки.
  2. Выполнить последовательное train / validation / test разбиение.
  3. Отделить predictors и target.
  4. Обучить train-only preprocessing и baseline-модели.
  5. Сравнить модели только по validation-метрикам.
  6. Проверить, что test-метрики не рассчитывались.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from sklearn.utils.validation import check_is_fitted

from traffic_forecasting.config import BASELINE_METRICS_PATH, DATETIME_COLUMN
from traffic_forecasting.data_loader import load_raw_data
from traffic_forecasting.features import build_feature_dataset
from traffic_forecasting.models import get_baseline_model_registry
from traffic_forecasting.pipeline import (
    save_baseline_metrics,
    train_and_evaluate_baselines,
)
from traffic_forecasting.preprocessing import prepare_model_inputs, split_chronologically

## **4.1. Подготовка набора признаков (Feature Dataset Preparation)**

In [ ]:
raw_data = load_raw_data()
feature_data = build_feature_dataset(raw_data)

display(Markdown("### **Размерность набора признаков (Feature Dataset Shape)**"))
display(pd.DataFrame({"rows": [len(feature_data)], "columns": [feature_data.shape[1]]}))

## **4.2. Хронологическое разбиение данных (Chronological Data Splitting)**

In [ ]:
train_data, validation_data, test_data = split_chronologically(feature_data)

split_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": len(split_data),
            "start": split_data[DATETIME_COLUMN].min(),
            "end": split_data[DATETIME_COLUMN].max(),
        }
        for split_name, split_data in (
            ("train", train_data),
            ("validation", validation_data),
            ("test", test_data),
        )
    ]
)

display(Markdown("### **Размеры и временные границы выборок (Split Sizes and Ranges)**"))
display(split_summary)

## **4.3. Подготовка модельных входов (Model Input Preparation)**

In [ ]:
(
    X_train,
    X_validation,
    X_test,
    y_train,
    y_validation,
    y_test,
    timestamps_train,
    timestamps_validation,
    timestamps_test,
) = prepare_model_inputs(train_data, validation_data, test_data)

input_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "X_rows": [len(X_train), len(X_validation), len(X_test)],
        "X_columns": [X_train.shape[1], X_validation.shape[1], X_test.shape[1]],
        "y_rows": [len(y_train), len(y_validation), len(y_test)],
        "timestamp_rows": [
            len(timestamps_train),
            len(timestamps_validation),
            len(timestamps_test),
        ],
    }
)

display(Markdown("### **Размерности модельных входов (Model Input Shapes)**"))
display(input_summary)

## **4.4. Состав baseline-моделей (Baseline Model Registry)**

In [ ]:
baseline_registry = get_baseline_model_registry()
model_registry_summary = pd.DataFrame(
    {
        "model": list(baseline_registry),
        "estimator": [type(model).__name__ for model in baseline_registry.values()],
    }
)

display(Markdown("### **Реестр baseline-моделей (Baseline Model Registry)**"))
display(model_registry_summary)

## **4.5. Обучение и validation-оценка (Training and Validation Evaluation)**

In [ ]:
validation_metrics, fitted_models, preprocessor = train_and_evaluate_baselines(
    X_train,
    X_validation,
    X_test,
    y_train,
    y_validation,
    y_test,
)
save_baseline_metrics(validation_metrics, BASELINE_METRICS_PATH)

validation_ranking = validation_metrics.sort_values("rmse").reset_index(drop=True)
display(Markdown("### **Validation-метрики baseline-моделей (Baseline Validation Metrics)**"))
display(validation_ranking.round(4))

## **4.6. Аудит методологических ограничений (Methodological Constraints Audit)**

In [ ]:
check_is_fitted(preprocessor)
for model in fitted_models.values():
    check_is_fitted(model)

methodology_audit = pd.DataFrame(
    {
        "check": [
            "train_precedes_validation",
            "validation_precedes_test",
            "metrics_are_validation_only",
            "validation_used_for_comparison",
            "test_metrics_remain_locked",
        ],
        "passed": [
            timestamps_train.max() < timestamps_validation.min(),
            timestamps_validation.max() < timestamps_test.min(),
            set(validation_metrics["split"]) == {"validation"},
            validation_metrics["used_for_model_comparison"].all(),
            "test" not in set(validation_metrics["split"]),
        ],
    }
)

display(Markdown("### **Результаты методологического аудита (Methodology Audit Results)**"))
display(methodology_audit)

assert bool(methodology_audit["passed"].all()), "Baseline methodology audit failed."

## **4.7. Сравнение baseline-моделей (Baseline Model Comparison)**

In [ ]:
best_validation_model = validation_ranking.iloc[[0]].copy()

display(Markdown("### **Лучшая baseline-модель по validation RMSE (Best Validation Baseline)**"))
display(best_validation_model)

## **4.8. Анализ и интерпретация результатов baseline-моделей (Analysis and Interpretation of Baseline Model Results)**

На этапе baseline-моделирования были обучены и оценены базовые регрессионные модели прогнозирования транспортной нагрузки на основе ранее подготовленных модельных признаков.

**Ключевые результаты:**
1. **Сформирован набор baseline-моделей для первичного сравнения.**
   В рамках этапа были рассмотрены модели `DummyRegressor`, `LinearRegression`, `Ridge`, `DecisionTreeRegressor`, `KNeighborsRegressor` и `SVR`. Данный набор включает наивную модель, линейные модели, простую нелинейную модель дерева решений, метрический алгоритм ближайших соседей и модель опорных векторов.
2. **Обучение моделей выполнено только на train-выборке.**
   Все baseline-модели обучались на обучающем подмножестве, сформированном с сохранением хронологического порядка наблюдений. Валидационная выборка использовалась только для оценки качества и сравнения моделей, что соответствует постановке задачи прогнозирования временного ряда и снижает риск утечки информации из будущих наблюдений.
3. **Оценка качества выполнена по основным регрессионным метрикам.**
   Для каждой модели были рассчитаны метрики `MAE`, `RMSE`, `MAPE` и `R²`. Метрики `MAE` и `RMSE` отражают абсолютную величину ошибки прогноза в единицах транспортного потока, `MAPE` показывает относительную ошибку в процентах, а `R²` характеризует долю объясненной дисперсии целевой переменной. Использование нескольких метрик позволяет оценивать качество моделей более полно, поскольку каждая из них отражает разные аспекты ошибки прогнозирования.
4. **Наилучший результат среди baseline-моделей показала модель дерева решений.**
   По результатам validation-оценки модель `DecisionTreeRegressor` продемонстрировала наиболее высокое качество среди рассмотренных baseline-алгоритмов. Это указывает на то, что в подготовленном наборе признаков присутствуют нелинейные зависимости, которые простая модель дерева решений способна учитывать лучше, чем наивная и линейные модели. Существенный вклад в качество прогноза могут вносить лаговые признаки и скользящие статистики, отражающие краткосрочную и суточную динамику транспортной нагрузки.
5. **Линейные модели и KNN сформировали дополнительный уровень сравнения.**
   Модели `LinearRegression` и `Ridge` показали сопоставимые результаты, что объясняется близкой линейной формой зависимости между признаками и целевой переменной. Модель `KNeighborsRegressor` также выступает важной baseline-точкой сравнения, поскольку использует близость объектов в пространстве признаков. При этом корректность применения KNN обеспечивается предварительной стандартизацией непрерывных числовых признаков в preprocessing-конвейере.
6. **Наивная модель подтвердила необходимость использования признаков.**
   `DummyRegressor`, формирующий прогноз на основе среднего значения целевой переменной, ожидаемо показал наиболее слабое качество. Его результат используется как нижняя граница качества прогнозирования. Существенное превосходство остальных моделей над наивным прогнозом подтверждает, что сформированные временные, календарные, погодные, лаговые и rolling-признаки содержат полезную информацию для решения задачи прогнозирования транспортной нагрузки.

**Итоговое методологическое резюме:** этап Baseline Models and Evaluation Metrics сформировал корректную исходную точку сравнения для дальнейшего исследования ансамблевых методов машинного обучения. Baseline-модели были обучены на train-выборке, оценены на validation-выборке и сопоставлены по метрикам `MAE`, `RMSE`, `MAPE` и `R²`. Test-выборка не использовалась для выбора модели, что сохраняет корректность последующей финальной оценки.
